# PodcastCleaner - Google Colab Pipeline

This notebook runs the PodcastCleaner audio cleaning pipeline on Google Colab,
using the GPU for ML-heavy processing stages (source separation, noise removal,
and optional transcription).

**Requirements:**
- Google Colab with GPU runtime (Runtime > Change runtime type > T4 or A100)
- Google Drive for persistent storage of models, config, and output

**How to use:**
1. Run cells top to bottom
2. Cell 5 verifies everything works before processing real audio
3. Set your input folder path in Cell 6
4. Run Cell 7 to process

Each section explains what the pipeline stage does and why it matters.

In [ ]:
# --- Environment Check ---
import subprocess, sys

# Verify Colab environment
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("WARNING: Not running in Colab. This notebook is designed for Colab.")

# Check GPU
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                           "--format=csv,noheader"],
                          capture_output=True, text=True)
if gpu_info.returncode == 0:
    print(f"GPU: {gpu_info.stdout.strip()}")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU")

## 2. Install Dependencies

Installs system packages (ffmpeg for audio encoding, libsndfile for audio I/O)
and the PodcastCleaner package with all its ML dependencies:

- **Demucs** (Meta/Facebook) — neural network for source separation
- **DeepFilterNet** — deep learning noise suppression
- **WhisperX** (optional) — speech-to-text with word-level timestamps
- **pyloudnorm** — loudness measurement and normalization

This takes ~2-3 minutes on first run. Subsequent runs in the same session are instant.

In [ ]:
# --- Install Dependencies ---
import subprocess, os, sys

# System packages
subprocess.run(["apt-get", "update", "-qq"], check=True,
               capture_output=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libsndfile1"],
               check=True, capture_output=True)
print("System deps installed (ffmpeg, libsndfile1)")

# Clone repo and install
REPO_URL = "https://github.com/TristanPetersDS/PodcastCleaner.git"
REPO_DIR = "/content/PodcastCleaner"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print(f"Cloned repo to {REPO_DIR}")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print(f"Updated repo at {REPO_DIR}")

# Install package with all optional deps (includes whisperx)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[all]",
                "-q"], check=True)
print("PodcastCleaner installed successfully")

## 3. Mount Google Drive & Cache Models

Mounts your Google Drive so processed audio and ML models persist across sessions.

**Model caching matters:** Demucs is ~500MB, DeepFilterNet ~50MB, WhisperX ~3GB.
Without caching, these re-download every time your Colab session resets. By storing
them on Drive, subsequent sessions start processing almost immediately.

**Directory structure on Drive:**
```
My Drive/PodcastCleaner/
├── models/          # Cached ML models (persistent)
├── output/          # Processed episode directories
└── config.yaml      # Your pipeline configuration
```

In [ ]:
# --- Mount Google Drive & Configure Cache ---
import os
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

# Base directory on Drive
DRIVE_BASE = Path("/content/drive/MyDrive/PodcastCleaner")
DRIVE_MODELS = DRIVE_BASE / "models"
DRIVE_OUTPUT = DRIVE_BASE / "output"

# Create directory structure
for d in [DRIVE_MODELS / "torch", DRIVE_MODELS / "huggingface",
          DRIVE_MODELS / "deepfilter", DRIVE_OUTPUT]:
    d.mkdir(parents=True, exist_ok=True)

# Point model caches to Drive (must be set BEFORE importing ML libraries)
os.environ["TORCH_HOME"] = str(DRIVE_MODELS / "torch")
os.environ["HF_HOME"] = str(DRIVE_MODELS / "huggingface")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# DeepFilterNet uses ~/.cache/DeepFilterNet — symlink to Drive
df_cache = Path.home() / ".cache" / "DeepFilterNet"
df_drive = DRIVE_MODELS / "deepfilter"
if not df_cache.exists():
    df_cache.parent.mkdir(parents=True, exist_ok=True)
    df_cache.symlink_to(df_drive)
    print(f"Symlinked DeepFilterNet cache -> {df_drive}")
elif df_cache.is_symlink():
    print(f"DeepFilterNet cache already symlinked -> {df_cache.resolve()}")
else:
    print(f"WARNING: {df_cache} exists and is not a symlink. "
          f"Models may not persist across sessions.")

print(f"TORCH_HOME:  {os.environ['TORCH_HOME']}")
print(f"HF_HOME:     {os.environ['HF_HOME']}")
print(f"Output dir:  {DRIVE_OUTPUT}")
print("Drive mounted and model cache configured!")

## 4. Configuration

Edit these settings to control the pipeline. Key parameters:

- **sample_rate**: 48kHz is broadcast standard — higher than CD quality (44.1kHz),
  preferred for spoken word because it preserves more high-frequency detail.
- **target_lufs**: -16 LUFS is the podcast loudness standard (Apple Podcasts, Spotify).
  LUFS measures *perceived* loudness, not just peak volume.
- **true_peak_dbtp**: -1.5 dBTP prevents digital clipping when encoding to MP3/AAC.
  Lossy codecs can overshoot peaks, so headroom is essential.
- **transcription**: Off by default. Enable if you want SRT subtitle files
  alongside your cleaned audio. Uses ~8-10GB VRAM with the large-v3 model.

In [ ]:
# --- Configuration ---
# Edit these settings, then run this cell to save config.

import yaml

ENABLE_TRANSCRIPTION = False  # Set True to generate SRT transcripts

config_dict = {
    "output_dir": str(DRIVE_OUTPUT),
    "preprocess": {
        "sample_rate": 48000,
        "channels": 1,
    },
    "separation": {
        "model": "htdemucs_ft",
        "device": "auto",
        "max_segment_minutes": 10,
    },
    "denoise": {
        "model": "DeepFilterNet3",
    },
    "transcription": {
        "enabled": ENABLE_TRANSCRIPTION,
        "model": "large-v3",
        "language": None,
        "device": "auto",
    },
    "normalization": {
        "target_lufs": -16.0,
        "true_peak_dbtp": -1.5,
    },
    "export": {
        "formats": ["mp3", "flac"],
        "mp3_bitrate": "320k",
        "sample_rate": 48000,
    },
}

# Save to Drive for persistence
config_path = DRIVE_BASE / "config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config_dict, f, default_flow_style=False)

print(f"Config saved to {config_path}")
print(f"Transcription: {'ON' if ENABLE_TRANSCRIPTION else 'OFF'}")
print(f"Output dir: {config_dict['output_dir']}")

## 5. Verify Installation

Three levels of verification before processing real audio:

1. **Unit tests** — 94 tests with mocked ML models (~30s). Confirms the package
   installed correctly and all dependencies resolve.
2. **System check** — Verifies ffmpeg, GPU detection, and ML library availability.
3. **Smoke test** — Generates a 10-second synthetic sine wave and runs it through
   the full pipeline (preprocess → separate → denoise → normalize → export).
   This catches real GPU/model issues that mocked tests can't find.

If all three pass, you're ready to process real audio.

In [ ]:
# --- Verification ---
import subprocess, sys, os

REPO_DIR = "/content/PodcastCleaner"

print("=" * 60)
print("STEP 1: Unit Tests (mocked ML, ~30s)")
print("=" * 60)
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-m", "not slow", "-v", "--tb=short"],
    cwd=REPO_DIR, capture_output=True, text=True
)
# Print last 20 lines (summary)
lines = result.stdout.strip().split("\n")
print("\n".join(lines[-20:]))
if result.returncode != 0:
    print("\nWARNING: Some tests failed. Check output above.")
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
else:
    print("\nAll unit tests passed!")

print("\n" + "=" * 60)
print("STEP 2: System Check")
print("=" * 60)
result = subprocess.run(
    [sys.executable, "-m", "podcast_cleaner.cli", "check"],
    cwd=REPO_DIR, capture_output=True, text=True
)
print(result.stdout)

print("\n" + "=" * 60)
print("STEP 3: Smoke Test (real ML models, ~1-2 min on GPU)")
print("=" * 60)

import tempfile, shutil
import numpy as np
import soundfile as sf
from pathlib import Path

# Generate 10s synthetic audio (440Hz sine + white noise)
sr = 48000
duration = 10.0
t = np.linspace(0, duration, int(sr * duration), endpoint=False)
amplitude = 10 ** (-20 / 20)
audio = (amplitude * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
# Add light noise to make denoising meaningful
audio += np.random.normal(0, 0.005, audio.shape).astype(np.float32)

# Set up episode dir in a temp location (not Drive, to keep it fast)
smoke_dir = Path(tempfile.mkdtemp()) / "00_Smoke-Test"
raw_dir = smoke_dir / "raw"
raw_dir.mkdir(parents=True)
sf.write(str(raw_dir / "smoke_test.wav"), audio, sr, subtype="FLOAT")

# Load config and run pipeline
from podcast_cleaner.config import load_config
from podcast_cleaner.cli import run_pipeline

config_path = DRIVE_BASE / "config.yaml"
config = load_config(str(config_path))
# Override output dir for smoke test
config["output_dir"] = str(smoke_dir.parent)

skip = ("download", "transcribe")
successes = run_pipeline(
    config=config,
    episode_dirs=[str(smoke_dir)],
    skip_stages=skip,
    resume=False,
    verbose=True,
)

# Verify outputs
final_dir = smoke_dir / "final"
if successes and final_dir.exists():
    outputs = list(final_dir.iterdir())
    print(f"\nSmoke test PASSED! Output files: {[f.name for f in outputs]}")

    # Print analysis stats
    report_path = smoke_dir / "analysis" / "audio_report.json"
    if report_path.exists():
        import json
        with open(report_path) as f:
            report = json.load(f)
        for stage, stats in report.items():
            lufs = stats.get("lufs", "N/A")
            snr = stats.get("snr_db", "N/A")
            print(f"  {stage}: LUFS={lufs}, SNR={snr}dB")
else:
    print("\nSmoke test FAILED. Check output above for errors.")

# Cleanup
shutil.rmtree(smoke_dir.parent, ignore_errors=True)

## 6. Select Audio Input

Point `INPUT_DIR` to your Google Drive folder containing raw audio files
(WAV, MP3, FLAC, M4A, OGG, OPUS, AAC).

The notebook **copies** files from your input folder into a fresh pipeline
directory structure. Your original files are never modified.

Each audio file becomes its own episode directory:
```
output/
├── 01_Episode-Name/
│   └── raw/episode.wav    ← copied from your input folder
├── 02_Another-Episode/
│   └── raw/episode.wav
```

In [ ]:
# --- Select Audio Input ---
# Set this to your Google Drive folder containing audio files.
INPUT_DIR = "/content/drive/MyDrive/your-podcast-folder"  # <-- EDIT THIS

import shutil
from pathlib import Path
from podcast_cleaner.utils import AUDIO_EXTENSIONS, ensure_dir, sanitize_filename

input_path = Path(INPUT_DIR)
if not input_path.exists():
    raise FileNotFoundError(
        f"Input directory not found: {INPUT_DIR}\n"
        f"Edit INPUT_DIR above to point to your Drive folder."
    )

# Find audio files
audio_files = sorted(
    f for f in input_path.iterdir()
    if f.is_file() and f.suffix.lower() in AUDIO_EXTENSIONS
)

if not audio_files:
    raise FileNotFoundError(
        f"No audio files found in {INPUT_DIR}\n"
        f"Supported formats: {', '.join(sorted(AUDIO_EXTENSIONS))}"
    )

# Create episode directories
episode_dirs = []
for idx, audio_file in enumerate(audio_files):
    safe_name = sanitize_filename(audio_file.stem)
    dirname = f"{idx + 1:02d}_{safe_name}"
    ep_dir = ensure_dir(DRIVE_OUTPUT / dirname)
    raw_dir = ensure_dir(ep_dir / "raw")
    dest = raw_dir / audio_file.name

    if not dest.exists():
        shutil.copy2(str(audio_file), str(dest))
        print(f"  Copied: {audio_file.name} -> {dirname}/raw/")
    else:
        print(f"  Exists: {dirname}/raw/{audio_file.name} (skipping copy)")

    episode_dirs.append(str(ep_dir))

print(f"\n{len(episode_dirs)} episode(s) ready for processing")

## 7. Run Pipeline

Processes each episode through the audio cleaning stages:

### Stage 1: Preprocess
Resamples audio to 48kHz mono. Podcast audio is mono by nature (speech from
one or two mics), and 48kHz is the broadcast standard sample rate. This
normalizes input from any format/rate into a consistent starting point.

### Stage 2: Separate (Demucs)
Uses Meta's **Demucs** neural network (`htdemucs_ft` — hybrid transformer,
fine-tuned) to separate vocals from background music and noise. The model
learned instrument separation from thousands of songs, and it works remarkably
well on podcasts — isolating speech from intro music, background noise, and
room reverb. This is the heaviest GPU stage (~6-8GB VRAM).

### Stage 3: Denoise (DeepFilterNet3)
Applies **DeepFilterNet3** noise suppression to the isolated vocals. While
Demucs removes music and distinct sounds, DeepFilterNet targets residual
ambient noise — air conditioning hum, computer fans, room tone. It processes
audio in 60-second chunks with crossfade overlap for seamless output.

### Stage 4: Transcribe (WhisperX) — Optional
If enabled, runs OpenAI's **Whisper** (via WhisperX) for speech-to-text with
word-level timestamps. Produces SRT subtitle files and JSON transcripts.
The large-v3 model uses ~8-10GB VRAM. Disabled by default.

### Stage 5: Normalize (pyloudnorm)
Adjusts loudness to **-16 LUFS** (the podcast standard for Apple Podcasts
and Spotify). LUFS measures *perceived* loudness — not just peak volume —
so your episodes sound consistent. Also applies true peak limiting at
-1.5 dBTP to prevent clipping in lossy codecs.

### Stage 6: Export
Encodes final audio to MP3 (320kbps for compatibility) and FLAC (lossless
for archival). This is the delivery-ready output.

---

Set `RESUME = True` if you're continuing after a session interruption.
Completed stages (marked by `.done` files) will be skipped automatically.

In [ ]:
# --- Run Pipeline ---
RESUME = False  # Set True to resume after session interruption

from podcast_cleaner.config import load_config
from podcast_cleaner.cli import run_pipeline

# Load config from Drive
config = load_config(str(DRIVE_BASE / "config.yaml"))

# Determine which stages to skip
skip = ["download"]  # Always skip download (we're using local files)
if not config.get("transcription", {}).get("enabled", False):
    skip.append("transcribe")

print(f"Processing {len(episode_dirs)} episode(s)")
print(f"Skipping stages: {skip}")
print(f"Resume mode: {RESUME}")
print(f"Output: {config['output_dir']}")
print("-" * 60)

successes = run_pipeline(
    config=config,
    episode_dirs=episode_dirs,
    skip_stages=tuple(skip),
    resume=RESUME,
    verbose=True,
)

print("=" * 60)
print(f"Complete: {len(successes)}/{len(episode_dirs)} episodes processed")
if len(successes) < len(episode_dirs):
    failed = set(episode_dirs) - set(successes)
    print(f"Failed episodes: {[Path(d).name for d in failed]}")
    print("Check processing.log in each episode dir for details.")

## 8. Results

Lists all output files and quality metrics. Your cleaned audio is in each
episode's `final/` directory on Google Drive.

The analysis report shows key audio quality metrics per stage:
- **LUFS**: Loudness (target: -16.0 for podcasts)
- **SNR**: Signal-to-noise ratio (higher = cleaner)
- **True Peak**: Maximum sample value in dBTP (should be < -1.0)
- **Duration**: Audio length in seconds

In [ ]:
# --- Results ---
import json
from pathlib import Path

print("=" * 60)
print("RESULTS")
print("=" * 60)

for ep_dir_str in successes:
    ep_dir = Path(ep_dir_str)
    ep_name = ep_dir.name
    print(f"\n{'─' * 60}")
    print(f"Episode: {ep_name}")
    print(f"{'─' * 60}")

    # List final output files
    final_dir = ep_dir / "final"
    if final_dir.exists():
        for f in sorted(final_dir.iterdir()):
            if f.is_file():
                size_mb = f.stat().st_size / (1024 * 1024)
                print(f"  {f.name} ({size_mb:.1f} MB)")

        # Check for transcripts
        transcript_dir = final_dir / "transcript"
        if transcript_dir.exists():
            for f in sorted(transcript_dir.iterdir()):
                print(f"  transcript/{f.name}")

    # Print analysis stats
    report_path = ep_dir / "analysis" / "audio_report.json"
    if report_path.exists():
        with open(report_path) as f:
            report = json.load(f)
        print(f"\n  {'Stage':<15} {'LUFS':>7} {'SNR':>7} {'Peak':>7} {'Duration':>10}")
        print(f"  {'─' * 50}")
        for stage, stats in report.items():
            lufs = f"{stats.get('lufs', 0):.1f}"
            snr = f"{stats.get('snr_db', 0):.1f}"
            peak = f"{stats.get('true_peak', 0):.1f}"
            dur = f"{stats.get('duration', 0):.1f}s"
            print(f"  {stage:<15} {lufs:>7} {snr:>7} {peak:>7} {dur:>10}")

print(f"\n{'=' * 60}")
print(f"All output saved to: {DRIVE_OUTPUT}")
print(f"Open Google Drive to access your files.")